# 01 · BIXI Data Preparation

**From raw BIXI trip records to auditable 15-minute station demand.**

This notebook forms the data-engineering stage of the *BIXI Smart Availability* portfolio project.
It takes official historical trip CSVs, constructs arrival/departure counts, aligns historical station names
to a 2024 reference, and creates time grids for dates when each station had recorded activity.

**Scope:** BIXI trip data only; 2022–2024 used for demand-model training. The subsequent notebooks
evaluate on held-out 2025 and 2026 data and investigate arrival-time availability.


## 1. Setup

Input RAW BIXI data files from 2022, 2023 and 2024:

```
data/raw/2022/2022dat.csv
data/raw/2023/2023dat.csv
data/raw/2024/2024dat.csv
```


In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

YEARS = (2022, 2023, 2024)
REFERENCE_YEAR = 2024
SLOT_SECONDS = 15 * 60
LOCAL_TZ = "America/Toronto"

candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next(
    (p for p in candidate_roots if (p / "data" / "raw").is_dir()), None
)

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

inputs = {year: raw_dir / str(year) / f"{year}dat.csv" for year in YEARS}
missing = [str(path) for path in inputs.values() if not path.is_file()]


con = duckdb.connect()
display(pd.DataFrame({
    "year": list(inputs),
    "input_csv": [str(p.relative_to(project_root)) for p in inputs.values()],
    "size_mb": [round(p.stat().st_size / 2**20, 1) for p in inputs.values()],
}))

## 2. Source-data quality

Trips with missing destination information cannot contribute to arrival counts, even when
their departure is known. 

In [ ]:
quality_sql = """
SELECT
    COUNT(*) AS trips,
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(STARTSTATIONNAME), '') IS NULL
    ) AS missing_start_station,
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(ENDSTATIONNAME), '') IS NULL
    ) AS missing_end_station,
    COUNT(*) FILTER (
        WHERE TRY_CAST(STARTTIMEMS AS BIGINT) IS NULL
    ) AS invalid_start_time,
    COUNT(*) FILTER (
        WHERE TRY_CAST(ENDTIMEMS AS BIGINT) IS NULL
    ) AS invalid_end_time
FROM read_csv(?, header=true, all_varchar=true)
"""

quality = []
for year, path in inputs.items():
    row = con.execute(quality_sql, [str(path)]).df()
    row.insert(0, "year", year)
    quality.append(row)
quality = pd.concat(quality, ignore_index=True)
display(quality)

## 3. Aggregate each trip as two timestamped events

Each valid trip start adds one departure; each valid trip end adds one arrival. Station labels are normalized for consistent joins, but geographic matching is still needed for renamed stations.

In [ ]:
aggregate_sql = """
COPY (
    WITH trips AS (
        SELECT
            LOWER(NULLIF(TRIM(STARTSTATIONNAME), '')) AS start_station,
            LOWER(NULLIF(TRIM(ENDSTATIONNAME), '')) AS end_station,
            TRY_CAST(STARTTIMEMS AS BIGINT) AS start_ms,
            TRY_CAST(ENDTIMEMS AS BIGINT) AS end_ms
        FROM read_csv(?, header=true, all_varchar=true)
    ),
    events AS (
        SELECT start_station AS station,
               CAST(FLOOR(start_ms / 900000.0) AS BIGINT) AS time_slot,
               0 AS arrivals, 1 AS departures
        FROM trips
        WHERE start_station IS NOT NULL AND start_ms IS NOT NULL
        UNION ALL
        SELECT end_station AS station,
               CAST(FLOOR(end_ms / 900000.0) AS BIGINT) AS time_slot,
               1 AS arrivals, 0 AS departures
        FROM trips
        WHERE end_station IS NOT NULL AND end_ms IS NOT NULL
    )
    SELECT station, time_slot,
           SUM(arrivals)::BIGINT AS arrivals,
           SUM(departures)::BIGINT AS departures
    FROM events
    GROUP BY station, time_slot
) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)
"""

for year, source in inputs.items():
    target = processed_dir / f"bixi_{year}_15min.parquet"
    con.execute(aggregate_sql, [str(source), str(target)])
    print(f"{year}: {target.name} ({target.stat().st_size / 2**20:.1f} MB)")

In [ ]:
event_check = []
for year in YEARS:
    source = processed_dir / f"bixi_{year}_15min.parquet"
    summary = con.execute("""
        SELECT COUNT(*) AS active_intervals,
               COUNT(DISTINCT station) AS stations,
               SUM(arrivals) AS arrivals,
               SUM(departures) AS departures
        FROM read_parquet(?)
    """, [str(source)]).df()
    summary.insert(0, "year", year)
    event_check.append(summary)
display(pd.concat(event_check, ignore_index=True))

## 4. Recover station locations from trips

The historical data identifies stations by name, not a stable station ID. We estimate each
station's location from the median of recorded coordinates in that year. The bounds below
exclude obviously invalid coordinates.

In [ ]:
locations_sql = """
WITH coordinates AS (
    SELECT LOWER(NULLIF(TRIM(STARTSTATIONNAME), '')) AS station,
           TRY_CAST(STARTSTATIONLATITUDE AS DOUBLE) AS lat,
           TRY_CAST(STARTSTATIONLONGITUDE AS DOUBLE) AS lon
    FROM read_csv(?, header=true, all_varchar=true)
    UNION ALL
    SELECT LOWER(NULLIF(TRIM(ENDSTATIONNAME), '')) AS station,
           TRY_CAST(ENDSTATIONLATITUDE AS DOUBLE) AS lat,
           TRY_CAST(ENDSTATIONLONGITUDE AS DOUBLE) AS lon
    FROM read_csv(?, header=true, all_varchar=true)
)
SELECT station, MEDIAN(lat) AS latitude, MEDIAN(lon) AS longitude,
       COUNT(*) AS coordinate_records
FROM coordinates
WHERE station IS NOT NULL AND lat BETWEEN 45 AND 46 AND lon BETWEEN -75 AND -72
GROUP BY station
"""

locations_by_year = []
for year, csv_path in inputs.items():
    data = con.execute(locations_sql, [str(csv_path), str(csv_path)]).df()
    data.insert(0, "year", year)
    locations_by_year.append(data)
locations = pd.concat(locations_by_year, ignore_index=True)
locations.to_parquet(processed_dir / "bixi_station_locations_2022_2024.parquet", index=False)
display(locations.groupby("year").agg(stations=("station", "nunique")))

## 5. Conservative station matching (reference: 2024)

We accept matching names only when positions agree within **30 m**. For renamed stations,
we accept a geographically closest candidate only within **5 m**, and only when the closest
reference is unambiguous. Duplicate targets and unresolved names are excluded.



In [ ]:
def haversine_m(lat_a, lon_a, lat_b, lon_b):
    lat_a, lon_a, lat_b, lon_b = map(np.radians, (lat_a, lon_a, lat_b, lon_b))
    dlat, dlon = lat_b - lat_a, lon_b - lon_a
    h = np.sin(dlat / 2) ** 2 + np.cos(lat_a) * np.cos(lat_b) * np.sin(dlon / 2) ** 2
    return 2 * 6_371_000 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))


reference = locations.loc[locations.year == REFERENCE_YEAR].copy()
reference = reference.drop_duplicates("station")
mapping_parts = []
for year in YEARS:
    source = locations.loc[locations.year == year].copy()
    if year == REFERENCE_YEAR:
        direct = source[["station"]].copy()
        direct["station_2024"] = direct["station"]
        direct["match_method"] = "reference_year"
        direct["distance_m"] = 0.0
    else:
        rows = []
        ref_lat, ref_lon = reference.latitude.to_numpy(), reference.longitude.to_numpy()
        ref_names = reference.station.to_numpy()
        by_name = reference.set_index("station")

        for item in source.itertuples(index=False):
            if item.station in by_name.index:
                ref_row = by_name.loc[item.station]
                gap = float(haversine_m(item.latitude, item.longitude,
                                        ref_row.latitude, ref_row.longitude))
                if gap <= 30:
                    rows.append((item.station, item.station, "name_and_location", gap))
                continue

            distances = haversine_m(item.latitude, item.longitude, ref_lat, ref_lon)
            nearest = np.argsort(distances)[:2]
            best = float(distances[nearest[0]])
            second = float(distances[nearest[1]]) if len(nearest) > 1 else np.inf
            if best <= 5 and second > best + 1:
                rows.append((item.station, ref_names[nearest[0]], "geographic", best))

        direct = pd.DataFrame(
            rows, columns=["station", "station_2024", "match_method", "distance_m"]
        )
        # Require one unique old name per reference station within the same year.
        direct = direct.loc[~direct.duplicated("station_2024", keep=False)].copy()

    direct.insert(0, "year", year)
    mapping_parts.append(direct)

mapping = pd.concat(mapping_parts, ignore_index=True)
assert not mapping.duplicated(["year", "station"]).any()
assert not mapping.duplicated(["year", "station_2024"]).any()
mapping.to_parquet(processed_dir / "bixi_station_mapping_2022_2024.parquet", index=False)
display(mapping.groupby(["year", "match_method"]).size().rename("stations").reset_index())

## 6. Construct a complete time grid for observed station-days

Trip event files contain only slots with recorded activity. For a station and local calendar day
with **at least one** recorded trip, add all 15-minute slots within that Montréal day. The
day can have 92, 96 or 100 slots around daylight-saving transitions.



In [ ]:
con.register("station_mapping", mapping)

matching_sql = """
COPY (
    SELECT m.station_2024 AS station, d.time_slot,
           SUM(d.arrivals)::BIGINT AS arrivals,
           SUM(d.departures)::BIGINT AS departures
    FROM read_parquet(?) d
    INNER JOIN station_mapping m
        ON d.station = m.station AND m.year = ?
    GROUP BY m.station_2024, d.time_slot
) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)
"""

for year in YEARS:
    target = processed_dir / f"bixi_{year}_matched.parquet"
    con.execute(matching_sql, [
        str(processed_dir / f"bixi_{year}_15min.parquet"), year, str(target)
    ])
    print("Matched:", target.name)

In [ ]:
def local_calendar(year):
    dates = pd.date_range(f"{year}-01-01", f"{year+1}-01-01", freq="D", inclusive="left")
    local_starts = dates.tz_localize(LOCAL_TZ)
    local_ends = (dates + pd.Timedelta(days=1)).tz_localize(LOCAL_TZ)
    return pd.DataFrame({
        "local_date": dates.date,
        "start_slot": (local_starts.asi8 // (SLOT_SECONDS * 10**9)),
        "end_slot": (local_ends.asi8 // (SLOT_SECONDS * 10**9)),
    })


complete_sql = """
COPY (
    WITH observed AS (
        SELECT station, time_slot, arrivals, departures,
               CAST(timezone('America/Toronto', to_timestamp(time_slot * 900)) AS DATE) AS local_date
        FROM read_parquet(?)
    ),
    active_days AS (
        SELECT DISTINCT station, local_date FROM observed
    ),
    grid AS (
        SELECT d.station, s.time_slot
        FROM active_days d
        INNER JOIN calendar_days c USING (local_date)
        CROSS JOIN generate_series(c.start_slot, c.end_slot - 1) s(time_slot)
    )
    SELECT g.station, g.time_slot,
           COALESCE(o.arrivals, 0)::BIGINT AS arrivals,
           COALESCE(o.departures, 0)::BIGINT AS departures,
           (o.time_slot IS NOT NULL) AS observed_trip
    FROM grid g
    LEFT JOIN observed o ON g.station = o.station AND g.time_slot = o.time_slot
) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)
"""

for year in YEARS:
    calendar = local_calendar(year)
    con.register("calendar_days", calendar)
    target = processed_dir / f"bixi_{year}_complete.parquet"
    con.execute(complete_sql, [
        str(processed_dir / f"bixi_{year}_matched.parquet"), str(target)
    ])
    print(f"{year}: {len(calendar)} local days; saved {target.name}")

## 7. Final validation and handoff

Verify that the completed grids conserve recorded arrivals/departures and contain no
duplicate station/time keys. These files feed **02 · Demand Forecasting**.

In [ ]:
checks = []
for year in YEARS:
    matched = processed_dir / f"bixi_{year}_matched.parquet"
    complete = processed_dir / f"bixi_{year}_complete.parquet"
    row = con.execute("""
        SELECT COUNT(*) AS intervals,
               COUNT(DISTINCT station) AS stations,
               SUM(arrivals) AS arrivals,
               SUM(departures) AS departures,
               SUM(CASE WHEN arrivals = 0 AND departures = 0 THEN 1 ELSE 0 END) AS zero_intervals,
               COUNT(*) - COUNT(DISTINCT (station, time_slot)) AS duplicate_keys
        FROM read_parquet(?)
    """, [str(complete)]).df().iloc[0].to_dict()
    totals = con.execute("""
        SELECT SUM(arrivals) AS arrivals, SUM(departures) AS departures
        FROM read_parquet(?)
    """, [str(matched)]).df().iloc[0]
    assert row["arrivals"] == totals["arrivals"]
    assert row["departures"] == totals["departures"]
    assert row["duplicate_keys"] == 0
    row["year"] = year
    checks.append(row)

summary = pd.DataFrame(checks)[
    ["year", "stations", "intervals", "arrivals", "departures", "zero_intervals", "duplicate_keys"]
]
display(summary)
print("All conservation and duplicate-key checks passed.")